In [3]:
import time

# используем bytearray, так как он выделяет один цельный кусок RAM на уровне C, в отличие от списков Python.
# Размер массива: 64 Мегабайта. Это больше, чем L3 кэш большинства процессоров.
ARRAY_SIZE = 64 * 1024 * 1024
data = bytearray(ARRAY_SIZE)

# 4096 байт — стандартный размер страницы памяти
# (или можно взять 64 байта — размер кэш-линии процессора).
STRIDE = 4096


def demo_ram_is_not_o1():
    print(f"Инициализация массива размером {ARRAY_SIZE / 1024 / 1024:.0f} MB...\n")

    # СЦЕНАРИЙ 1: Последовательный доступ (Cache-friendly)
    print("1. Запуск последовательного обхода памяти (Row-major)...")
    start_time = time.time()

    # Проходим те же 64 млн итераций, строго друг за другом
    for i in range(ARRAY_SIZE):
        data[i] = 1

    seq_time = time.time() - start_time
    print(f"Время выполнения: {seq_time:.4f} секунд\n")

    # СЦЕНАРИЙ 2: Доступ с прыжками (Cache-unfriendly / Cache Misses)
    print("2. Запуск обхода с шагом (разрушение префетчинга)...")
    start_time = time.time()

    # Делаем ровно столько же итераций, но скачем по памяти с шагом STRIDE.
    # Внутренний цикл нужен, чтобы в сумме обойти весь массив,
    # сначала элементы: 0, 4096, 8192..., потом 1, 4097, 8193...
    for offset in range(STRIDE):
        for i in range(offset, ARRAY_SIZE, STRIDE):
            data[i] = 2

    stride_time = time.time() - start_time
    print(f"Время выполнения: {stride_time:.4f} секунд\n")

    print("Результат")
    print(f"Сложность обхода в обоих случаях: O(N)")
    print(f"Количество операций: {ARRAY_SIZE}")
    print(f"Разница в скорости: последовательный быстрее в {stride_time / seq_time:.2f} раз!")
    print("Вывод: 'O(1) доступ к RAM' — это академическая иллюзия.")


if __name__ == '__main__':
    demo_ram_is_not_o1()

Инициализация массива размером 64 MB...

1. Запуск последовательного обхода памяти (Row-major)...
Время выполнения: 2.0383 секунд

2. Запуск обхода с шагом (разрушение префетчинга)...
Время выполнения: 3.6042 секунд

Результат
Сложность обхода в обоих случаях: O(N)
Количество операций: 67108864
Разница в скорости: последовательный быстрее в 1.77 раз!
Вывод: 'O(1) доступ к RAM' — это академическая иллюзия.


In [4]:
import os
import time
import random

FILE_SIZE_MB = 100
BLOCK_SIZE = 4096
TOTAL_WRITES = 20000  # Сделаем 20 000 небольших записей

FILE_NAME = "ssd_write_demo.dat"


def setup_file():
    """Создаем файл 100 МБ, забитый нулями (имитация существующей БД)."""
    print(f"Подготовка файла БД на {FILE_SIZE_MB} MB...")
    with open(FILE_NAME, "wb") as f:
        f.write(b'\x00' * (FILE_SIZE_MB * 1024 * 1024))
    # Заставляем ОС физически сбросить данные из кэша на диск
    os.sync()


def demo_ssd_write_amplification():
    setup_file()

    payload = b'\xAF' * 256  # Представим, что мы обновляем строку в БД (256 байт)
    max_offset = (FILE_SIZE_MB * 1024 * 1024) - len(payload)

    print("\n--- СЦЕНАРИЙ 1: Классическая реляционная БД (Random In-place Updates) ---")
    print(f"Обновляем {TOTAL_WRITES} случайных строк прямо внутри файла...")

    start_time = time.time()
    with open(FILE_NAME, "r+b") as f:
        for _ in range(TOTAL_WRITES):
            # Прыгаем в случайное место файла
            offset = random.randint(0, max_offset)
            f.seek(offset)
            # Перезаписываем маленьки кусок (256 байт)
            f.write(payload)
            f.flush()
        # fsync нужен, чтобы сбросить буфер контроллера. Без этого ОС "считерит"
        # и оставит всё в памяти (Page Cache).
        os.fsync(f.fileno())

    random_time = time.time() - start_time
    print(f"Случайная перезапись заняла: {random_time:.4f} секунд")

    print("\n--- СЦЕНАРИЙ 2: Брокер сообщений / LSM-Tree (Sequential Append-only) ---")
    print(f"Дописываем те же {TOTAL_WRITES} обновлений СТРОГО В КОНЕЦ лога...")

    start_time = time.time()
    with open(FILE_NAME, "ab") as f:  # ВАЖНО: Режим 'ab' - append (добавление в конец)
        for _ in range(TOTAL_WRITES):
            f.write(payload)
            f.flush()
        os.fsync(f.fileno())

    append_time = time.time() - start_time
    print(f"Последовательное добавление заняло: {append_time:.4f} секунд")

    print("\n--------- ИТОГИ ---------")
    print(f"Объем полезной записи одинаковый: {(TOTAL_WRITES * 256) / 1024 / 1024:.2f} MB")
    print(f"Последовательный Append-only быстрее случайного In-place обновления в {random_time / append_time:.2f} раз!")

    os.remove(FILE_NAME)


if __name__ == '__main__':
    demo_ssd_write_amplification()

Подготовка файла БД на 100 MB...

--- СЦЕНАРИЙ 1: Классическая реляционная БД (Random In-place Updates) ---
Обновляем 20000 случайных строк прямо внутри файла...
Случайная перезапись заняла: 0.2583 секунд

--- СЦЕНАРИЙ 2: Брокер сообщений / LSM-Tree (Sequential Append-only) ---
Дописываем те же 20000 обновлений СТРОГО В КОНЕЦ лога...
Последовательное добавление заняло: 0.0339 секунд

--------- ИТОГИ ---------
Объем полезной записи одинаковый: 4.88 MB
Последовательный Append-only быстрее случайного In-place обновления в 7.61 раз!
